## Section 11 — Isolation Forest
**Spacecraft Telemetry Anomaly Detection — Phase 2**

- Unsupervised model trained on clean normal data only
- Anomaly scores computed over the full injected dataset
- Labels used only for evaluation, never for training

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings, time, json
warnings.filterwarnings('ignore')
np.random.seed(42)

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score,
    precision_recall_curve
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa'})
os.makedirs('plots_v2', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Libraries loaded.')

### 11.1 Load & Prepare Data

In [ ]:
# Load the labelled anomaly dataset from Section 10
tel = pd.read_csv('data/telemetry_with_anomalies.csv', parse_dates=['timestamp'])
tel = tel.sort_values(['parameter','timestamp']).reset_index(drop=True)

tel_wide = tel.pivot_table(index='timestamp', columns='parameter',
                            values='value', aggfunc='mean').reset_index()
tel_wide.columns.name = None
tel_wide = tel_wide.sort_values('timestamp').reset_index(drop=True)

label_wide = tel.pivot_table(index='timestamp', columns='parameter',
                              values='is_anomaly', aggfunc='max').reset_index()
label_wide.columns.name = None
label_wide = label_wide.sort_values('timestamp').reset_index(drop=True)

param_cols = [c for c in tel_wide.columns if c != 'timestamp']
tel_wide[param_cols]   = tel_wide[param_cols].ffill().bfill()
label_wide[param_cols] = label_wide[param_cols].ffill().fillna(0)

y_true = label_wide[param_cols].max(axis=1).astype(int).values
X_all  = tel_wide[param_cols].values

print(f'Dataset loaded: {tel.shape}')
print(f'Wide-format shape: {X_all.shape}')
print(f'Anomaly rows (wide, with ffill): {y_true.sum()} / {len(y_true)}')

### 11.2 Train / Test Split & Scaling

In [ ]:
# Train on clean normal rows from the first 70% of timestamps
# Simulates having historical clean data before anomalies started appearing
cutoff     = int(0.70 * len(X_all))
train_mask = (y_true[:cutoff] == 0)   # keep only rows with no anomaly
X_train    = X_all[:cutoff][train_mask]

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit ONLY on clean training rows
X_all_scaled   = scaler.transform(X_all)          # apply same scale to full dataset

print(f'Training rows (normal only, first 70%): {len(X_train_scaled):,}')
print(f'Scoring rows  (full dataset)          : {len(X_all_scaled):,}')
print()
print('Rule: scaler fitted on clean rows only. Labels never seen during training.')

# Row-level anomaly type for per-type recall analysis
type_wide = tel.pivot_table(index='timestamp', columns='parameter',
                             values='anomaly_type', aggfunc='first').reset_index()
type_wide.columns.name = None
type_wide = type_wide.sort_values('timestamp').reset_index(drop=True)
def row_type(row):
    for v in row.values:
        if pd.notna(v) and v != 'none': return v
    return 'none'
type_col = type_wide.drop(columns='timestamp').apply(row_type, axis=1)

### 11.3 Train Isolation Forest

In [ ]:
# n_estimators=200: 200 random trees — more trees = more stable scores
# contamination=0.03: tells model to expect ~3% anomalies (matches our injection rate)
# In production: use contamination='auto' and tune threshold based on operator tolerance
t0 = time.time()
ifo = IsolationForest(n_estimators=200, contamination=0.03,
                       random_state=42, n_jobs=-1)
ifo.fit(X_train_scaled)   # only sees normal data — learns what 'normal' looks like
t_train = time.time() - t0

print(f'Model trained in {t_train:.2f}s')
print(f'Total trees  : {len(ifo.estimators_)}')
print(f'Features     : {ifo.n_features_in_}')
print()
print('How it works: normal points cluster in dense regions')
print('and need many splits to isolate (long path) -> low anomaly score.')
print('Anomalies sit alone -> few splits needed -> high anomaly score.')

### 11.4 Score & Predict

In [ ]:
t0 = time.time()
scores_if = ifo.decision_function(X_all_scaled)  # raw score: negative = more anomalous
y_pred_if = ifo.predict(X_all_scaled)             # +1 = normal, -1 = anomaly
t_score   = time.time() - t0

# Convert to binary 0/1 (1 = anomaly)
y_pred_if_bin = (y_pred_if == -1).astype(int)
scores_if_inv = -scores_if   # flip: higher score = more anomalous

prec_if = precision_score(y_true, y_pred_if_bin, zero_division=0)
rec_if  = recall_score(y_true, y_pred_if_bin, zero_division=0)
f1_if   = f1_score(y_true, y_pred_if_bin, zero_division=0)
auc_if  = roc_auc_score(y_true, scores_if_inv)
cm_if   = confusion_matrix(y_true, y_pred_if_bin)
tn, fp, fn, tp = cm_if.ravel()
fpr_if, tpr_if, _ = roc_curve(y_true, scores_if_inv)

print('ISOLATION FOREST RESULTS')
print(f'  Precision : {prec_if:.4f}  (of rows flagged, how many are truly anomalous)')
print(f'  Recall    : {rec_if:.4f}  (of all anomalies, how many were caught)')
print(f'  F1 Score  : {f1_if:.4f}')
print(f'  AUC-ROC   : {auc_if:.4f}  (threshold-independent — primary metric)')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'  Train time: {t_train:.2f}s | Score time: {t_score:.3f}s')

### 11.5 Evaluation Plots

In [ ]:
thresh = np.percentile(scores_if_inv, 97)
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
fig.suptitle('Isolation Forest — Evaluation', fontsize=12, fontweight='bold')

# Score distribution
axes[0].hist(scores_if_inv[y_true==0], bins=50, alpha=0.7, color='#1f77b4',
             label='Normal', density=True)
axes[0].hist(scores_if_inv[y_true==1], bins=50, alpha=0.7, color='#d62728',
             label='Anomaly', density=True)
axes[0].axvline(thresh, color='black', ls='--', lw=1.5, label='Threshold')
axes[0].set_title('Score Distribution')
axes[0].set_xlabel('Anomaly Score (higher = more anomalous)')
axes[0].legend(fontsize=8)
# Good model: red peak (anomaly) to the right of blue peak (normal)

# Confusion matrix
sns.heatmap(cm_if, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Normal','Pred Anomaly'],
            yticklabels=['True Normal','True Anomaly'],
            ax=axes[1], cbar=False, annot_kws={'size':12, 'weight':'bold'})
axes[1].set_title('Confusion Matrix')

# ROC curve
axes[2].plot(fpr_if, tpr_if, color='#1f77b4', lw=2, label=f'AUC = {auc_if:.3f}')
axes[2].plot([0,1],[0,1], 'k--', lw=1)
axes[2].fill_between(fpr_if, tpr_if, alpha=0.1, color='#1f77b4')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate (Recall)')
axes[2].set_title('ROC Curve')
axes[2].legend(fontsize=9)
# Curve should hug the top-left corner. AUC > 0.9 = excellent.

plt.tight_layout()
plt.savefig('plots_v2/11_iforest_eval.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.6 Per Anomaly-Type Recall

In [ ]:
# How well does IF detect each type of anomaly?
type_results_if = []
for atype in ['point', 'contextual', 'collective']:
    mask = type_col == atype
    if mask.sum() > 0:
        yt = (type_col[mask] != 'none').astype(int).values
        yp = y_pred_if_bin[mask.values]
        type_results_if.append({
            'Anomaly Type': atype,
            'Total Rows'  : int(mask.sum()),
            'Detected'    : int((yp == 1).sum()),
            'Recall'      : round(recall_score(yt, yp, zero_division=0), 3),
            'Precision'   : round(precision_score(yt, yp, zero_division=0), 3)
        })

df_types = pd.DataFrame(type_results_if)
display(df_types)
# Expected: Point > Contextual > Collective
# Collective anomalies (drifts/oscillations across sequences) are hardest for snapshot models

In [ ]:
# Save results for Section 13 comparison
ifo_results = {
    'model'      : 'Isolation Forest',
    'precision'  : round(prec_if, 4),
    'recall'     : round(rec_if, 4),
    'f1'         : round(f1_if, 4),
    'auc'        : round(auc_if, 4),
    'train_time' : round(t_train, 3),
    'score_time' : round(t_score, 3),
    'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
    'by_type': type_results_if,
    'fpr': fpr_if.tolist(),
    'tpr': tpr_if.tolist()
}
with open('models/iforest_results.json', 'w') as f:
    json.dump(ifo_results, f, indent=2)
print('Saved: models/iforest_results.json')